# End-to-End App Demo
This notebook walks through an end-to-end flow of the app: request  retrieval  ranking  output formatting.

## 1) Setup
Load environment variables and configure paths for local execution.

In [ ]:
from pathlib import Path
import os
import json

# Project root (adjust if this notebook is moved)
ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
BACKEND_DIR = ROOT / "backend"

print("Root:", ROOT)
print("Data dir:", DATA_DIR)
print("Backend dir:", BACKEND_DIR)

## 2) Load Sample Inputs
Pick a sample user query and any optional filters.

In [ ]:
query = "something italian with tomato"
filters = {
    "diet": "vegetarian",
    
}

print("Query:", query)
print("Filters:", filters)

## 3) Retrieval
Use the vector store to retrieve candidate recipes.

In [ ]:
# NOTE: Adjust imports to your app's modules if needed
from backend.app.vector_db.recipe_vector_store import RecipeVectorStore

# Initialize store (assumes index exists)
store = RecipeVectorStore()

candidates = store.retrieve(query, top_k=20, filters=filters)
print("Candidates:", len(candidates))
print(candidates[:3])

## 4) Ranking
Rank the retrieved recipes based on relevance and constraints.

In [ ]:
# NOTE: Adjust imports to your app's ranking utilities if needed
from backend.app.models.recipe import Recipe
from backend.app.db.repository import RecipeRepository

# Example: hydrate recipe objects if candidates are ids
repo = RecipeRepository()

if candidates and isinstance(candidates[0], (str, int)):
    recipe_objs = [repo.get_recipe_by_id(rid) for rid in candidates]
else:
    recipe_objs = candidates

# Simple sort stub (replace with your ranking logic)
ranked = sorted(
    [r for r in recipe_objs if r],
    key=lambda r: getattr(r, "score", 0),
    reverse=True
)

print("Ranked:", len(ranked))
print(ranked[:3])

## 5) Output Formatting
Format the top results for UI or API response.

In [ ]:
def recipe_to_output(recipe, rank):
    return {
        "rank": rank,
        "id": getattr(recipe, "id", None),
        "title": getattr(recipe, "title", None),
        "summary": getattr(recipe, "summary", None),
        "time_minutes": getattr(recipe, "time_minutes", None),
        "score": getattr(recipe, "score", None),
    }

top_k = 5
output = [recipe_to_output(r, i + 1) for i, r in enumerate(ranked[:top_k])]

print(json.dumps(output, indent=2))

## 6) End-to-End Result
This is the final payload your app can return from the API.

In [ ]:
final_payload = {
    "query": query,
    "filters": filters,
    "results": output,
}

print(json.dumps(final_payload, indent=2))